# 04 — Frozen model evaluation

**NOT RUN**

Prepared but intentionally not executed. Run manually in Google Colab.

Experiments E08/E09. Requires notebook 03 checkpoints and fixed split hashes.

Upload the project ZIP when prompted. Set only the flags for the stage you intend to run. Model stages need `INSTALL_MODEL_PACKAGES=True`; core stages need only PyYAML. Back up datasets/checkpoints/results to Drive manually before the Colab session ends.


In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys
import zipfile

PROJECT_DIR = Path.cwd() / "certificate-recovery"
ARCHIVE = Path.cwd() / "Dependency_Certificate_Recovery_Foundation.zip"
if not PROJECT_DIR.exists():
    if not ARCHIVE.exists():
        from google.colab import files
        uploaded = files.upload()
        archives = [name for name in uploaded if name.endswith(".zip")]
        if len(archives) != 1:
            raise ValueError("Upload exactly one project ZIP")
        ARCHIVE = Path.cwd() / archives[0]
    with zipfile.ZipFile(ARCHIVE) as bundle:
        destination = Path.cwd().resolve()
        for member in bundle.infolist():
            if not (destination / member.filename).resolve().is_relative_to(destination):
                raise ValueError("Unsafe archive path")
        bundle.extractall(destination)
if not (PROJECT_DIR / "src" / "cert_recovery").exists():
    raise FileNotFoundError("Project source directory missing")
sys.path.insert(0, str(PROJECT_DIR / "src"))

INSTALL_CORE_PACKAGES = False
INSTALL_MODEL_PACKAGES = False
if INSTALL_CORE_PACKAGES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")])
if INSTALL_MODEL_PACKAGES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements-colab.txt")])
if importlib.util.find_spec("yaml") is None:
    raise RuntimeError("Set INSTALL_CORE_PACKAGES=True and rerun setup; no model packages are needed for the core.")

from cert_recovery.config import load_config
config = load_config(PROJECT_DIR / "config" / "config.yaml")
print("Prepared but intentionally not executed. Run manually in Google Colab.")


Freeze training, model revision, threshold candidates and annotation decisions before setting SELECTION_FROZEN. Test results must not guide further model selection. This compares a pretrained NLI proxy to a trained dependency classifier; the tasks differ. Neither probability is a safety certificate.

In [ ]:
from cert_recovery.model_pipeline import run_pretrained_baseline, run_model_evaluation

RUN_EVALUATION = False
SELECTION_FROZEN = False
if RUN_EVALUATION:
    if not SELECTION_FROZEN:
        raise RuntimeError("Freeze selection before opening held-out results")
    config["execution"]["allow_model_inference"] = True
    pretrained = run_pretrained_baseline(config, manual=True, include_test=True, selection_frozen=True)
    finetuned = run_model_evaluation(config, manual=True, selection_frozen=True)
    print({"pretrained_proxy": pretrained["test"]["classification"],
           "fine_tuned_dependency": finetuned["test"]["classification"]})
else:
    print("NOT RUN: model evaluation and comparison")


Report false-negative dependency errors, source/domain slices and calibration. Do not compare headline F1 without understanding annotation validity.